# Day 14 — One-Stage Object Detectors

**Agri-AI Bootcamp** · Phase 4 of 5: *Object Detection & Yield* · Days 13–16

> Implement the NMS sorting loop from scratch on mock detected-weed boxes.

**Runtime:** `Runtime -> Change runtime type -> T4 GPU` before you start.

**How to use this notebook:** run the cells top to bottom. The *Worked solution*
section is complete and runnable; the *Your turn* section at the end is where you
experiment. Datasets that need a Kaggle/Roboflow key are simulated here with the
same columns and shapes, and the real download command is shown in a comment —
swap it in when you have the credentials. Example numbers are illustrative.

## 1. Explained

YOLO looks at the whole image once and outputs many candidate boxes with class scores. Several boxes often land on the same object. NMS keeps the highest-confidence box, discards others whose IoU with it is above a threshold, then repeats.

### 🌾 Agriculture example

One weed patch gets five overlapping boxes; with a 0.5 IoU threshold, NMS leaves one box per weed.

> **Watch out:** A threshold that’s too low merges neighbouring plants; too high leaves duplicates.

## 2. Setup

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
import matplotlib.patches as patches

## 3. Data

In [ ]:
# ---- A detector's raw output: many overlapping boxes on 3 weed patches ------
torch.manual_seed(0)
TRUE_WEEDS = torch.tensor([[ 40,  50, 120, 130],
                           [200,  70, 280, 150],
                           [120, 220, 200, 300]], dtype=torch.float32)
boxes, scores = [], []
for k, w in enumerate(TRUE_WEEDS):
    for _ in range(5):                            # 5 noisy proposals per weed
        jit = torch.randn(4) * 8
        boxes.append(w + jit)
        scores.append(float(torch.rand(1) * 0.4 + 0.55))
BOXES  = torch.stack(boxes)
SCORES = torch.tensor(scores)
print(f"{len(BOXES)} raw proposals for {len(TRUE_WEEDS)} real weed patches")

## 4. Starter code from the course

Reference only — the shortest form of today's idea. It assumes variables that the
next section defines properly, so read it, don't run it. Section 5 is the runnable
version.

```python
def nms(boxes,scores,thr=0.5):
    order=scores.argsort(descending=True).tolist(); keep=[]
    while order:
        i=order.pop(0); keep.append(i)
        order=[j for j in order if iou(boxes[i],boxes[j])<thr]
    return keep
```

## 5. Worked solution

In [ ]:
def iou(a, b):
    x1 = torch.max(a[0], b[0]); y1 = torch.max(a[1], b[1])
    x2 = torch.min(a[2], b[2]); y2 = torch.min(a[3], b[3])
    inter = (x2-x1).clamp(min=0) * (y2-y1).clamp(min=0)
    union = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / union if union > 0 else torch.tensor(0.)

# ---- Non-Maximum Suppression, written out step by step ----------------------
def nms(boxes, scores, thr=0.5, verbose=False):
    order = scores.argsort(descending=True).tolist()   # 1. sort by confidence
    keep = []
    while order:
        i = order.pop(0)                               # 2. take the best left
        keep.append(i)
        dropped = [j for j in order if iou(boxes[i], boxes[j]) >= thr]
        if verbose and dropped:
            print(f"  keep {i} (score {scores[i]:.2f}) -> suppress {dropped}")
        order = [j for j in order if j not in dropped] # 3. drop its duplicates
    return keep

print("NMS at IoU 0.5:")
keep = nms(BOXES, SCORES, 0.5, verbose=True)
print(f"\n{len(BOXES)} proposals -> {len(keep)} detections (truth: {len(TRUE_WEEDS)})")

# ---- Threshold is a trade-off, not a constant -------------------------------
print("\nthr   boxes kept   comment")
for thr in [0.1, 0.3, 0.5, 0.7, 0.9]:
    k = nms(BOXES, SCORES, thr)
    note = ("merges neighbouring plants" if len(k) < len(TRUE_WEEDS)
            else "about right" if len(k) == len(TRUE_WEEDS) else "duplicates left over")
    print(f"{thr:<5} {len(k):>10}   {note}")

# ---- Cross-check against torchvision ----------------------------------------
from torchvision.ops import nms as tv_nms
mine = sorted(nms(BOXES, SCORES, 0.5))
theirs = sorted(tv_nms(BOXES, SCORES, 0.5).tolist())
print(f"\nmine {mine}\ntorchvision {theirs}\nmatch: {mine == theirs}")

# ---- Class-aware NMS: never suppress a weed with an apple -------------------
labels = torch.tensor([0,0,0,0,0, 1,1,1,1,1, 0,0,0,0,0])   # 0=weed, 1=crop
def nms_per_class(boxes, scores, labels, thr=0.5):
    out = []
    for c in labels.unique():
        m = (labels == c).nonzero().flatten()
        out += [int(m[i]) for i in nms(boxes[m], scores[m], thr)]
    return out
print("\nper-class NMS keeps:", sorted(nms_per_class(BOXES, SCORES, labels)))
print("Suppressing across classes would delete a crop box that merely overlaps a weed.")

# ---- Before / after ----------------------------------------------------------
fig, ax = plt.subplots(1, 2, figsize=(11, 4.6))
for a, idxs, title in [(ax[0], range(len(BOXES)), f"raw: {len(BOXES)} boxes"),
                       (ax[1], keep, f"after NMS: {len(keep)} boxes")]:
    a.set_xlim(0, 340); a.set_ylim(340, 0); a.set_facecolor("#eaf3e6")
    for b in TRUE_WEEDS:
        a.add_patch(patches.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1],
                    fill=False, ec="green", lw=2.4))
    for i in idxs:
        b = BOXES[i]
        a.add_patch(patches.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1],
                    fill=False, ec="crimson", lw=1.3, ls="--"))
    a.set_title(title)
plt.tight_layout(); plt.show()

## 6. Your turn

1. Add a 6th proposal that overlaps two weeds at IoU 0.45 each. Does NMS keep it, and should it?
2. Implement Soft-NMS: decay scores by (1 - IoU) instead of deleting boxes.
3. Time your loop against `torchvision.ops.nms` on 2000 random boxes.

In [ ]:
# TODO 1: the ambiguous proposal

# TODO 2: Soft-NMS
def soft_nms(boxes, scores, sigma=0.5, cut=0.3):
    s = scores.clone(); keep = []
    order = s.argsort(descending=True).tolist()
    while order:
        i = order.pop(0); keep.append(i)
        for j in list(order):
            o = iou(boxes[i], boxes[j])
            s[j] = s[j] * torch.exp(-o**2 / sigma)      # decay, do not delete
            if s[j] < cut: order.remove(j)
        order.sort(key=lambda k: -s[k])
    return keep, s
k2, s2 = soft_nms(BOXES, SCORES)
print("soft-NMS keeps", len(k2), "boxes")

# TODO 3: timing on 2000 boxes

## 7. Dataset and references

**Dataset:** [Hand-write 4 to 6 box coordinates for apples or weeds](https://colab.research.google.com/#create=true)

**References**

- [You Only Look Once (paper)](https://arxiv.org/abs/1506.02640)

**Course material:** Redmon et al.: You Only Look Once

---
**Next:** Day 15 — The Ultralytics Framework on T4